In [0]:
import requests
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, 
    DoubleType, ArrayType, TimestampType
)

years = ["2020", "2021", "2022", "2023", "2024", "2025"]


tasks_df = spark.createDataFrame([(y,) for y in years], ["year"]).repartition(6)

output_schema = "year STRING, raw_json STRING, status STRING"

json_schema = StructType([
    StructField("latitude", DoubleType()),
    StructField("longitude", DoubleType()),
    StructField("hourly", StructType([
        StructField("time", ArrayType(StringType())),
        StructField("temperature_2m", ArrayType(DoubleType())),
        StructField("relative_humidity_2m", ArrayType(DoubleType())),
        StructField("wind_speed_10m", ArrayType(DoubleType()))
    ]))
])

def fetch_weather_batch(iterator):
    session = requests.Session()
    
    for pdf in iterator:
        results = []
        for year in pdf['year']:
            url = "https://archive-api.open-meteo.com/v1/archive"
            params = {
                "latitude": 52.4064,
                "longitude": 16.9252,
                "start_date": f"{year}-01-01", 
                "end_date": f"{year}-12-31",
                "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m",
                "timezone": "Europe/Warsaw"
            }
            try:
                resp = session.get(url, params=params, timeout=30)
                resp.raise_for_status() 
                results.append({"year": year, "raw_json": resp.text, "status": "Success"})
            except requests.exceptions.RequestException as e:
                results.append({"year": year, "raw_json": None, "status": str(e)})
                
        yield pd.DataFrame(results)

raw_json_df = tasks_df.mapInPandas(fetch_weather_batch, schema=output_schema)

successful_df = raw_json_df.filter(F.col("status") == "Success")

parsed_df = successful_df.withColumn("data", F.from_json(F.col("raw_json"), json_schema)).select("data.*")

exploded_df = parsed_df.select(
    F.col("latitude"), 
    F.col("longitude"), 
    F.explode(
        F.arrays_zip(
            F.col("hourly.time"), 
            F.col("hourly.temperature_2m"), 
            F.col("hourly.relative_humidity_2m"), 
            F.col("hourly.wind_speed_10m")
        )
    ).alias("reading")
)

master_df = exploded_df.select(
    F.col("latitude"),
    F.col("longitude"),
    F.col("reading.time").cast("timestamp").alias("timestamp"),
    F.col("reading.temperature_2m").alias("temperature_celsius"),
    F.col("reading.relative_humidity_2m").alias("humidity_percent"),
    F.col("reading.wind_speed_10m").alias("wind_speed_kmh")
)

display(master_df)

In [0]:
from pyspark.sql.window import Window

gold_df = master_df.withColumn("temperature_celsius", F.col("temperature_celsius").cast("double"))

window_spec = Window.partitionBy("latitude", "longitude").orderBy("timestamp")
 
gold_df = gold_df.withColumn("prev_temp", F.lag("temperature_celsius").over(window_spec))

gold_df = gold_df.withColumn(
    "temp_change", 
    F.abs(F.col("temperature_celsius") - F.col("prev_temp"))
).withColumn(
    "is_anomaly",
    F.when(F.col("temp_change") > 3.0, True).otherwise(False)
)

In [0]:
full_weather_data = gold_df.withColumn(
    "temp_change", F.round("temp_change", 2) 
)

stats = full_weather_data.select(F.stddev("temp_change")).collect()
print(f"Sigma: {stats[0][0]}")

full_weather_data.createOrReplaceTempView("full_weather_data_temp")

Sigma: 0.6246129715720692


In [0]:
%sql

CREATE TABLE IF NOT EXISTS full_weather_data (
    latitude DOUBLE,
    longitude DOUBLE,
    timestamp TIMESTAMP,
    temperature_celsius DOUBLE,
    humidity_percent DOUBLE,
    wind_speed_kmh DOUBLE,
    prev_temp DOUBLE,
    temp_change DOUBLE,
    is_anomaly BOOLEAN
) USING DELTA;

INSERT OVERWRITE full_weather_data
SELECT 
    latitude,
    longitude,
    timestamp,
    temperature_celsius,
    humidity_percent,
    wind_speed_kmh,
    prev_temp,
    temp_change,
    is_anomaly
FROM 
    full_weather_data_temp;

CREATE OR REPLACE VIEW weather_gold_anomalies_view AS
SELECT 
    timestamp,
    temperature_celsius,
    prev_temp,
    temp_change
FROM 
    full_weather_data
WHERE 
    is_anomaly = TRUE;

SELECT * FROM weather_gold_anomalies_view ORDER BY temp_change DESC;
